# LIC-DSF — all Output sheets

Load Inputs 1–8 from the template via `lic_dsf` loaders, compute each
user-facing Output, and display the panels.

| Excel Output | Computed from |
|---|---|
| 1-1 / 1-2 | `lic_dsf.dsa` baseline panels |
| 2-1 / 3-1 | `lic_dsf.stress` external B-tests |
| 2-2 / 3-2 | public B1 GDP (+ baseline) |
| 4-1 / 4-2 | `lic_dsf.realism` |
| 5-1 / 5-2 / 6 / 7 | `lic_dsf.rating` + `lic_dsf.scenario` |
| Summary / Database | metadata + indicator extract |

Workbook: `data/lic-dsf-template-2025-08-12.xlsx`.

See `docs/README.md`, `docs/01-excel-map.md`, and `docs/11-scenario.md` (Output 6).

## 0. Load inputs and build books

In [5]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from fastpyxl import load_workbook

from lic_dsf.dsa import (
    external_dsa_panel,
    load_core,
    public_dsa_panel,
)
from lic_dsf.pv import load_input7_residual_params
from lic_dsf.rating import (
    ChartDataRegistry,
    MarketFinancingInputs,
    RiskRatingSummary,
    assess_market_financing,
    compute_mechanical_ratings,
    load_ci_summary,
    load_trigger_flags,
    market_panel,
    moderate_panel,
    risk_summary_panel,
)
from lic_dsf.realism import (
    fiscal_adjustment_panel,
    fiscal_multiplier_panel,
    forecast_error_panel,
    invest_growth_panel,
    load_capital_assumptions,
    load_imported_data,
    place_in_lic_histogram,
    placement_summary,
    projected_three_year_adjustment,
    rebase_ratio_to_outturn_gdp,
    three_year_fiscal_adjustment,
)
from lic_dsf.scenario import (
    CustomizedScenarioSpec,
    ProbabilityAssumptions,
    probability_panel,
    register_custom_path,
)
from lic_dsf.stress import (
    load_input6_standard,
    run_b1_gdp_public,
    run_standard_external_stress,
    stress_external_panel,
    stress_public_panel,
)

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "demo":
    REPO_ROOT = REPO_ROOT.parent

WORKBOOK = REPO_ROOT / "data" / "lic-dsf-template-2025-08-12.xlsx"

pd.set_option("display.max_columns", 16)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

WORKBOOK

PosixPath('/home/sravan/excel-grapher/lic-dsf/data/lic-dsf-template-2025-08-12.xlsx')

In [6]:
# Inputs 3–5 / 4 / 8 → Macro + Ext + baseline books via load_core;
# Input 6 / 7 shock + ResFin params; Classification / CI Summary thresholds.
macro, external, ext_base, pub_base = load_core(WORKBOOK)

first_proj = macro.inputs.first_projection_year
hist_years = list(range(first_proj - 3, first_proj))
proj_years = list(range(first_proj, first_proj + 11))
display_years = hist_years + proj_years[:5]

ci = load_ci_summary(WORKBOOK)
input6 = load_input6_standard(WORKBOOK)
residual = load_input7_residual_params(WORKBOOK)

{
    "country": ci.country,
    "ifs": ci.country_code,
    "first_projection_year": first_proj,
    "dcc": ci.dcc.value,
    "ci_score": round(ci.ci_score, 6),
    "thresholds": ci.thresholds.as_dict(),
    "input6_gdp_shock_sd": input6.gdp_shock_sd,
    "resfin_shares": (
        residual.external_mlt_share,
        residual.domestic_mlt_share,
        residual.domestic_st_share,
    ),
}

{'country': 'Ghana',
 'ifs': 652,
 'first_projection_year': 2024,
 'dcc': 'Medium',
 'ci_score': 2.739886,
 'thresholds': {'pv_debt_to_exports': 180.0,
  'pv_debt_to_gdp': 40.0,
  'debt_service_to_exports': 15.0,
  'debt_service_to_revenue': 18.0,
  'public_pv_debt_to_gdp': 55.0},
 'input6_gdp_shock_sd': 1.0,
 'resfin_shares': (0.4343925896763784,
  0.2275655436226405,
  0.33804186670098113)}

## Output 1-1 — External DSA

In [7]:
out_1_1 = external_dsa_panel(ext_base)
out_1_1.loc[:, display_years]

,2021,2022,2023,2024,2025,2026,2027,2028
PV of PPG external debt / GDP,NaN,NaN,42.9240,44.8846,43.1514,41.2265,40.1941,38.6043
PV of PPG external debt / exports,NaN,NaN,105.1065,109.0647,103.4689,99.5562,98.4192,95.2973
PV of PPG external debt / revenue,NaN,NaN,259.0434,247.7666,237.3607,221.1705,213.7632,206.4453
PPG debt service / exports,10.5848,12.3440,14.7305,17.1681,16.5074,15.1705,16.2756,18.5608
PPG debt service / revenue,23.5345,31.3280,36.3045,39.0015,37.8686,33.7022,35.3500,40.2087
External GFN (USD),"1,556.9076","1,488.5526","1,337.0348","2,360.2673","2,091.5041","2,101.2626","2,600.8647","3,504.1136"


## Output 1-2 — Public DSA

In [8]:
out_1_2 = public_dsa_panel(pub_base)
out_1_2.loc[:, display_years]

,2021,2022,2023,2024,2025,2026,2027,2028
Public sector debt / GDP,70.4386,72.9075,74.2261,69.7520,65.1962,61.8445,59.5446,57.2213
PPG external debt / GDP,44.9115,49.1634,49.2827,50.0070,47.6584,45.7766,44.7547,42.8747
PV of public debt / GDP,25.5271,23.7441,71.0383,67.1667,62.4801,58.6475,55.9502,53.6994
PV of public debt / revenue+grants,138.6104,121.2321,411.7692,356.2547,335.3506,309.1418,293.6842,284.3530
Debt service / revenue+grants,93.3344,114.6024,127.0609,126.3839,92.4878,85.3944,82.5395,87.6262
Public GFN / GDP,21.0472,22.8186,24.3585,22.4982,15.0812,13.9981,13.7347,14.7661


## Output 2-1 / 3-1 — External stress

Standard external B-tests from Input 6 + Input 7 residual financing.
(A1 historical / B2 primary-balance / tailored C* not yet runners.)

In [9]:
external_stress = run_standard_external_stress(
    macro, external, input6, residual
)
list(external_stress)

['B1_GDP', 'B3_Exports', 'B4_OtherFlows', 'B5_FX', 'B6_Combo']

In [10]:
def _ratio_row(label: str, series: pd.Series, years: list[int]) -> pd.Series:
    return series.reindex(years).rename(label)


def stress_table_for_indicator(getter_name: str, years: list[int]) -> pd.DataFrame:
    rows = [_ratio_row("Baseline", getattr(ext_base, getter_name)(), years)]
    labels = {
        "B1_GDP": "B1. Real GDP growth",
        "B3_Exports": "B3. Exports",
        "B4_OtherFlows": "B4. Other flows",
        "B5_FX": "B5. Depreciation",
        "B6_Combo": "B6. Combination of B1-B5",
    }
    for sid, label in labels.items():
        book = external_stress[sid]
        rows.append(_ratio_row(label, getattr(book, getter_name)(), years))
    return pd.DataFrame(rows)


out_3_1_blocks = {
    "PV of debt-to-GDP": stress_table_for_indicator(
        "pv_ppg_external_to_gdp", proj_years
    ),
    "PV of debt-to-exports": stress_table_for_indicator(
        "pv_ppg_external_to_exports", proj_years
    ),
    "Debt service-to-exports": stress_table_for_indicator(
        "ppg_debt_service_to_exports", proj_years
    ),
    "Debt service-to-revenue": stress_table_for_indicator(
        "ppg_debt_service_to_revenue", proj_years
    ),
}
out_2_1 = out_3_1_blocks  # chart series = same paths
for name, frame in out_3_1_blocks.items():
    print(f"\n=== {name} ===")
    print(frame.to_string())
out_3_1_blocks["PV of debt-to-GDP"]


=== PV of debt-to-GDP ===
                            2024    2025    2026    2027    2028    2029    2030    2031    2032    2033    2034
Baseline                 44.8846 43.1514 41.2265 40.1941 38.6043 37.6952 35.4235 32.9897 31.9558 31.4151 31.2078
B1. Real GDP growth      44.8846 46.9546 46.6468 45.4787 43.6799 42.6512 40.0808 37.3271 36.1573 35.5455 35.3109
B3. Exports              44.8846 48.0971 55.6099 62.5539 68.2709 73.9874 77.0337 78.3145 79.5957 80.1006 79.8755
B4. Other flows          44.8846 46.7521 48.1803 46.7100 44.6855 43.3395 40.1575 36.4264 34.2544 32.7103 31.6211
B5. Depreciation         44.8846 54.7005 52.2604 50.9516 48.9364 47.7839 44.9042 41.8191 40.5085 39.8231 39.5603
B6. Combination of B1-B5 44.8846 51.7962 57.9999 61.1725 63.3390 65.8794 65.9054 64.6368 64.2211 63.7026 63.0127

=== PV of debt-to-exports ===
                             2024     2025     2026     2027     2028     2029     2030     2031     2032     2033     2034
Baseline                 10

,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034
Baseline,44.8846,43.1514,41.2265,40.1941,38.6043,37.6952,35.4235,32.9897,31.9558,31.4151,31.2078
B1. Real GDP growth,44.8846,46.9546,46.6468,45.4787,43.6799,42.6512,40.0808,37.3271,36.1573,35.5455,35.3109
B3. Exports,44.8846,48.0971,55.6099,62.5539,68.2709,73.9874,77.0337,78.3145,79.5957,80.1006,79.8755
B4. Other flows,44.8846,46.7521,48.1803,46.7100,44.6855,43.3395,40.1575,36.4264,34.2544,32.7103,31.6211
B5. Depreciation,44.8846,54.7005,52.2604,50.9516,48.9364,47.7839,44.9042,41.8191,40.5085,39.8231,39.5603
B6. Combination of B1-B5,44.8846,51.7962,57.9999,61.1725,63.3390,65.8794,65.9054,64.6368,64.2211,63.7026,63.0127


## Output 2-2 / 3-2 — Public stress

Public B1 GDP with residual financing (other public B*/C* not yet runners).

In [11]:
public_b1 = run_b1_gdp_public(macro, external, input6, residual)

out_3_2_pv_gdp = pd.DataFrame(
    [
        _ratio_row("Baseline", pub_base.pv_public_debt_to_gdp(), proj_years),
        _ratio_row(
            "B1. Real GDP growth", public_b1.pv_public_debt_to_gdp(), proj_years
        ),
    ]
)
out_3_2_ds_rev = pd.DataFrame(
    [
        _ratio_row(
            "Baseline", pub_base.debt_service_to_revenue_grants(), proj_years
        ),
        _ratio_row(
            "B1. Real GDP growth",
            public_b1.debt_service_to_revenue_grants(),
            proj_years,
        ),
    ]
)
out_2_2 = {"PV of Debt-to-GDP": out_3_2_pv_gdp, "Debt Service-to-Revenue": out_3_2_ds_rev}
print(stress_public_panel(public_b1).loc[:, proj_years[:5]].to_string())
out_3_2_pv_gdp

                                     2024        2025        2026        2027        2028
Public sector debt / GDP          69.7520     72.3888     73.4734     72.8067     72.2519
PV of public debt / GDP           67.1667     69.4333     69.8627     68.7502     68.2794
Debt service / revenue+grants    126.3839     92.4878     87.7751     88.3005     96.0198
Public GFN (LCU)              45,533.9567 39,639.0524 45,503.1524 52,486.9787 64,570.8892


,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034
Baseline,67.1667,62.4801,58.6475,55.9502,53.6994,51.4136,49.3587,47.9421,46.9933,46.1724,45.3455
B1. Real GDP growth,67.1667,69.4333,69.8627,68.7502,68.2794,67.4807,66.8920,67.0764,67.8312,68.7001,69.2672


## Output 4-1 — Forecast Error (Realism 1)

Uses `Imported data` vintages (linked from Inputs / prior DSA dumps).

In [12]:
imported = load_imported_data(WORKBOOK)
print(imported.country, imported.country_code, "series:", len(imported.series))

prior_ppg = imported.get("D_PPG_GDP", 2019) or imported.get("D_PPG_GDP", "2019")
current_ext_debt = pub_base.ppg_external_debt_to_gdp()

if prior_ppg is not None:
    prior_gdp = imported.get("NGDPD", prior_ppg.vintage_year)
    curr_gdp = imported.get("NGDPD", imported.current_vintage_year)
    if prior_gdp is not None and curr_gdp is not None:
        rebased_prior = rebase_ratio_to_outturn_gdp(
            prior_gdp.values, curr_gdp.values, prior_ppg.values
        )
    else:
        rebased_prior = prior_ppg.values
    overlap = sorted(
        set(rebased_prior.dropna().index) & set(current_ext_debt.dropna().index)
    )
    out_4_1 = forecast_error_panel(
        current_ext_debt.reindex(overlap),
        rebased_prior.reindex(overlap),
    )
else:
    out_4_1 = pd.DataFrame({"note": ["No D_PPG_GDP vintage in Imported data"]})

out_4_1

Ghana 652 series: 26


,2017,2018,2019,2020,2021,2022,2023,2024,...,2027,2028,2029,2030,2031,2032,2033,2034
Current DSA debt / GDP,23.4554,23.7030,29.9102,34.7496,44.9115,49.1634,49.2827,50.0070,...,44.7547,42.8747,41.7026,40.0721,37.4220,36.2181,35.5573,34.9295
Prior DSA debt / GDP,37.4823,38.6612,40.6809,38.4503,36.2600,34.3510,33.4275,32.9582,...,30.6683,30.3636,30.2969,30.0824,29.5049,28.3598,27.3018,26.3584
Forecast error (prior − current),14.0269,14.9581,10.7707,3.7007,-8.6515,-14.8124,-15.8551,-17.0488,...,-14.0864,-12.5111,-11.4056,-9.9897,-7.9171,-7.8583,-8.2555,-8.5711


## Output 4-2 — Realism tools (Realism 2–4)

In [ ]:
pd_pct = pub_base.primary_deficit_to_gdp()
projected_adj = projected_three_year_adjustment(pd_pct, first_proj)
placement = place_in_lic_histogram(projected_adj)
out_4_2_fiscal_adj = fiscal_adjustment_panel(pd_pct, first_proj)

print("3-year adjustment:")
print(three_year_fiscal_adjustment(pd_pct).loc[first_proj : first_proj + 2].to_string())
print("Projected placement:")
print(placement_summary(placement).to_string())
out_4_2_fiscal_adj.head(8)

In [ ]:
pb_pct = 100.0 * macro.primary_balance() / macro.gdp_lcu().replace(0.0, pd.NA)
out_4_2_multiplier = fiscal_multiplier_panel(
    pb_pct, macro.real_gdp_growth(), first_proj
)
out_4_2_multiplier.loc[proj_years[:8], :]

In [ ]:
# Ig/Y path from Realism 3 sheet (FAD / prior-vintage feed until Macro owns Ig)
wb = load_workbook(WORKBOOK, data_only=True, read_only=True)
try:
    ws = wb["Realism 3 - Invest-Growth"]
    r3_years = {
        int(ws.cell(19, c).value): c
        for c in range(3, 20)
        if isinstance(ws.cell(19, c).value, (int, float))
    }
    ig_curr = pd.Series(
        {
            y: float(ws.cell(21, c).value)
            for y, c in r3_years.items()
            if isinstance(ws.cell(21, c).value, (int, float))
        },
        dtype=float,
    )
finally:
    wb.close()

cap = load_capital_assumptions(WORKBOOK)
out_4_2_invest = invest_growth_panel(
    ig_curr, macro.real_gdp_growth().reindex(ig_curr.index), cap
)
out_4_2_invest

## Chart Data registry → Outputs 5 / 6 / 7

In [ ]:
registry = ChartDataRegistry()

registry.register_series(
    "pv_debt_to_gdp",
    "baseline",
    ext_base.pv_ppg_external_to_gdp().reindex(proj_years),
    is_baseline=True,
)
registry.register_series(
    "pv_debt_to_exports",
    "baseline",
    ext_base.pv_ppg_external_to_exports().reindex(proj_years),
    is_baseline=True,
)
registry.register_series(
    "debt_service_to_exports",
    "baseline",
    ext_base.ppg_debt_service_to_exports().reindex(proj_years),
    is_baseline=True,
)
registry.register_series(
    "debt_service_to_revenue",
    "baseline",
    ext_base.ppg_debt_service_to_revenue().reindex(proj_years),
    is_baseline=True,
)
registry.register_series(
    "public_pv_debt_to_gdp",
    "baseline",
    pub_base.pv_public_debt_to_gdp().reindex(proj_years),
    is_baseline=True,
)

for sid, book in external_stress.items():
    registry.register_series(
        "pv_debt_to_gdp",
        sid,
        book.pv_ppg_external_to_gdp().reindex(proj_years),
        is_shock=True,
    )
    registry.register_series(
        "pv_debt_to_exports",
        sid,
        book.pv_ppg_external_to_exports().reindex(proj_years),
        is_shock=True,
    )
    registry.register_series(
        "debt_service_to_exports",
        sid,
        book.ppg_debt_service_to_exports().reindex(proj_years),
        is_shock=True,
    )
    registry.register_series(
        "debt_service_to_revenue",
        sid,
        book.ppg_debt_service_to_revenue().reindex(proj_years),
        is_shock=True,
    )

registry.register_series(
    "public_pv_debt_to_gdp",
    "B1_GDP",
    public_b1.pv_public_debt_to_gdp().reindex(proj_years),
    is_shock=True,
)

register_custom_path(
    registry,
    indicator="pv_debt_to_gdp",
    values=ext_base.pv_ppg_external_to_gdp().reindex(proj_years),
    spec=CustomizedScenarioSpec(name="Customized", short_name="custom"),
)

mechanical = compute_mechanical_ratings(
    registry, ci.thresholds, years=proj_years
)
{
    "external": mechanical.external.label,
    "fiscal": mechanical.fiscal.label,
    "overall": mechanical.overall.label,
    "ext_baseline_breach": mechanical.external_baseline_breach,
    "ext_shock_breach": mechanical.external_shock_breach,
    "fis_baseline_breach": mechanical.fiscal_baseline_breach,
    "fis_shock_breach": mechanical.fiscal_shock_breach,
    "paths": len(registry.paths),
}

## Output 5-1 — Moderate risk

In [ ]:
out_5_1 = moderate_panel(
    mechanical_external=mechanical.external,
    baseline_pv_gdp=ext_base.pv_ppg_external_to_gdp(),
    threshold_pv_gdp=ci.thresholds.pv_debt_to_gdp,
    rating_years=proj_years,
)
out_5_1

## Output 5-2 — Market module

In [ ]:
trigger = load_trigger_flags(WORKBOOK, ci.country_code)
gfn = pub_base.public_gfn_to_gdp().reindex(list(range(first_proj, first_proj + 3)))
market = assess_market_financing(
    MarketFinancingInputs(
        market_access=bool(trigger.market_lic) if trigger else False,
        gfn_to_gdp=gfn,
        embi_spread=None,
    )
)
out_5_2 = market_panel(market)
print("Trigger market_lic:", None if trigger is None else trigger.market_lic)
out_5_2

## Output 6 — Probability approach

In [ ]:
mx_sid = max(
    external_stress,
    key=lambda s: float(
        external_stress[s].pv_ppg_external_to_gdp().reindex(proj_years).max()
    ),
)
out_6 = probability_panel(
    {
        "baseline": ext_base.pv_ppg_external_to_gdp().reindex(proj_years),
        "mx_shock": external_stress[mx_sid]
        .pv_ppg_external_to_gdp()
        .reindex(proj_years),
    },
    ci.thresholds.pv_debt_to_gdp,
    indicator="pv_debt_to_gdp",
    assumptions=ProbabilityAssumptions(bandwidth=0.1),
)
print("Most extreme shock:", mx_sid)
out_6

## Output 7 — Risk rating summary

In [ ]:
summary = RiskRatingSummary(
    mechanical=mechanical,
    thresholds=ci.thresholds,
    dcc=ci.dcc,
    ci_score=ci.ci_score,
    moderate_granularity=str(out_5_1.loc["Space to absorb shock", "Output 5-1"]),
)
out_7 = risk_summary_panel(summary)
out_7

In [ ]:
judged = summary.apply_judgement(
    final_external=summary.mechanical.external,
    final_overall=summary.mechanical.overall,
    note="(demo) finals echo mechanical",
)
risk_summary_panel(judged)

## Output - Summary / Database extract

In [ ]:
out_summary = pd.Series(
    {
        "Country": ci.country,
        "IFS Code": ci.country_code,
        "First Year of Projection": first_proj,
        "Composite Indicator": ci.ci_score,
        "Debt Carrying Capacity": ci.dcc.value,
        "Mechanical external": mechanical.external.label,
        "Mechanical fiscal": mechanical.fiscal.label,
        "Mechanical overall": mechanical.overall.label,
        **{f"Threshold {k}": v for k, v in ci.thresholds.as_dict().items()},
    },
    name="Output Summary",
).to_frame()
out_summary

In [ ]:
def _db_rows(code: str, series: pd.Series) -> pd.DataFrame:
    s = series.reindex(proj_years).dropna()
    return pd.DataFrame(
        {
            "Indicator": code,
            "Country.IMF Country Code": ci.country_code,
            "Year": s.index.astype(int),
            "Value": s.to_numpy(dtype=float),
        }
    )


out_database = pd.concat(
    [
        _db_rows("DPPVNPV_GDP", ext_base.pv_ppg_external_to_gdp()),
        _db_rows("DPPVNPV_BX", ext_base.pv_ppg_external_to_exports()),
        _db_rows("TDS_BX", ext_base.ppg_debt_service_to_exports()),
        _db_rows("TDS_REV", ext_base.ppg_debt_service_to_revenue()),
        _db_rows("DU_NPV_GDP", pub_base.pv_public_debt_to_gdp()),
        _db_rows("DU_GDP", pub_base.public_sector_debt_to_gdp()),
    ],
    ignore_index=True,
)
out_database.head(20)

## Catalog

In [ ]:
outputs = {
    "Output 1-1 External DSA": out_1_1,
    "Output 1-2 Public DSA": out_1_2,
    "Output 3-1 PV/GDP": out_3_1_blocks["PV of debt-to-GDP"],
    "Output 3-1 PV/exports": out_3_1_blocks["PV of debt-to-exports"],
    "Output 3-1 DS/exports": out_3_1_blocks["Debt service-to-exports"],
    "Output 3-1 DS/revenue": out_3_1_blocks["Debt service-to-revenue"],
    "Output 3-2 Public PV/GDP": out_3_2_pv_gdp,
    "Output 3-2 Public DS/rev": out_3_2_ds_rev,
    "Output 4-1 Forecast Error": out_4_1,
    "Output 4-2 Fiscal adjustment": out_4_2_fiscal_adj,
    "Output 4-2 Fiscal multiplier": out_4_2_multiplier,
    "Output 4-2 Invest-growth": out_4_2_invest,
    "Output 5-1 Moderate": out_5_1,
    "Output 5-2 Market": out_5_2,
    "Output 6 Probability": out_6,
    "Output 7 Risk rating": out_7,
    "Output - Summary": out_summary,
    "Output Database": out_database,
}
pd.DataFrame(
    {
        "output": list(outputs),
        "shape": [getattr(v, "shape", None) for v in outputs.values()],
    }
)